# Embedding 実験（ベースライン複製 + 多角的検証）

- **ベース**: train_baseline.ipynb と同じ 38 特徴 + 各 embedding の PCA 特徴
- **Embedding**: 4 種類 × 2 パターン（PCA 8 / PCA 16）
  - `movie_info_embeddings.pkl` / `movie_info_embeddings_large.pkl`
  - `movie_title_info_embeddings.pkl` / `movie_title_info_embeddings_large.pkl`
- **検証（多角的・リークなし）**:
  1. **時系列CV**: 学習 = `review_year < val_year`、検証 = `review_year == val_year`（未来のデータで学習しない）
  2. **GroupKFold（ベースラインv2 風）**: 映画単位で分割（同じ映画は tr か val のどちらか一方のみ）
  3. **PCA**: 各 fold の**学習データ（tr）だけで** fit し、val はその PCA で transform（検証情報が PCA に漏れない）
- **バックグラウンド実行**: ノートブックを閉じてもよい場合は、ターミナルで
  `nohup python run_embedding_experiments.py > outputs/embedding_experiments.log 2>&1 &`
  結果は `outputs/embedding_experiment_results.csv` に保存されます。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES, BASELINE_LGB_PARAMS

OUTPUT_DIR = "outputs"
EMBEDDINGS_DIR = Path(OUTPUT_DIR) / "embeddings"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
# データ読み込み（38 特徴）
train, test = get_baseline_data()
y = train["target"].values
print(f"Train: {len(train):,}, Test: {len(test):,}, Features: {len(BASELINE_FEATURES)}")

Train: 653,507, Test: 40,716, Features: 38


In [4]:
# 検証方法 1: 時系列CV（2013〜2016 を val）
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

# 検証方法 2: GroupKFold by movie（ベースラインv2 の未知用検証と同じ考え方）
GROUP_KFOLD_N = 5
gkf = GroupKFold(n_splits=GROUP_KFOLD_N)
groups_movie = train["rotten_tomatoes_link"].values
group_splits = list(gkf.split(train, y, groups=groups_movie))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
print(f"GroupKFold: {GROUP_KFOLD_N} folds (group=映画)")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
GroupKFold: 5 folds (group=映画)


In [6]:
# 4 種類の embedding 設定（各 2 パターン: PCA 8, PCA 16）
EMBEDDING_CONFIGS = [
    {"name": "movie_info", "path": EMBEDDINGS_DIR / "movie_info_embeddings.pkl", "loader": "movie_info"},
    {"name": "movie_info_large", "path": EMBEDDINGS_DIR / "movie_info_embeddings_large.pkl", "loader": "movie_info"},
    {"name": "movie_title_info", "path": EMBEDDINGS_DIR / "movie_title_info_embeddings.pkl", "loader": "title_info"},
    {"name": "movie_title_info_large", "path": EMBEDDINGS_DIR / "movie_title_info_embeddings_large.pkl", "loader": "title_info"},
]
PCA_PATTERNS = [8, 16]

def load_embeddings(config):
    path = config["path"]
    if not path.exists():
        return None
    if config["loader"] == "movie_info":
        from lib.openai_embeddings import load_movie_info_embeddings
        return load_movie_info_embeddings(path=path)
    from lib.openai_embeddings import load_movie_title_info_embeddings
    return load_movie_title_info_embeddings(path=path)

def _merge_embeddings(df, emb_df):
    """df に embedding を left join し、emb 行列を返す。"""
    emb_cols = [c for c in emb_df.columns if c != "rotten_tomatoes_link"]
    merged = df[["rotten_tomatoes_link"]].merge(
        emb_df[["rotten_tomatoes_link"] + emb_cols], on="rotten_tomatoes_link", how="left"
    )
    return merged[emb_cols].fillna(0).astype(np.float32).values

In [6]:
# 各 fold の「学習データだけで」PCA を fit し、検証データはその PCA で transform（リーク防止）
def run_time_series_cv(train_df, test_df, base_features, y, time_splits, emb_df, n_comp, prefix):
    """時系列CV: 学習=review_year<val_year, 検証=その年。PCA は各 fold の tr だけで fit。"""
    E_train = _merge_embeddings(train_df, emb_df)
    n_comp = min(n_comp, E_train.shape[1])
    pca_names = [f"{prefix}_{i}" for i in range(n_comp)]
    scores = []
    for tr_idx, val_idx in time_splits:
        pca = PCA(n_components=n_comp, random_state=42).fit(E_train[tr_idx])
        train_pca = pca.transform(E_train)
        X_tr = pd.concat([
            train_df[base_features].iloc[tr_idx].reset_index(drop=True),
            pd.DataFrame(train_pca[tr_idx], columns=pca_names),
        ], axis=1)
        X_val = pd.concat([
            train_df[base_features].iloc[val_idx].reset_index(drop=True),
            pd.DataFrame(train_pca[val_idx], columns=pca_names),
        ], axis=1)
        model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
        model.fit(X_tr, y[tr_idx], eval_set=[(X_val, y[val_idx])], callbacks=[lgb.early_stopping(30, verbose=False)])
        scores.append(roc_auc_score(y[val_idx], model.predict_proba(X_val)[:, 1]))
    return float(np.mean(scores)), float(np.std(scores))

def run_group_kfold_cv(train_df, test_df, base_features, y, group_splits, emb_df, n_comp, prefix):
    """GroupKFold: 同じ映画は tr か val のどちらか。PCA は各 fold の tr だけで fit。"""
    E_train = _merge_embeddings(train_df, emb_df)
    n_comp = min(n_comp, E_train.shape[1])
    pca_names = [f"{prefix}_{i}" for i in range(n_comp)]
    scores = []
    for tr_idx, val_idx in group_splits:
        pca = PCA(n_components=n_comp, random_state=42).fit(E_train[tr_idx])
        train_pca = pca.transform(E_train)
        X_tr = pd.concat([
            train_df[base_features].iloc[tr_idx].reset_index(drop=True),
            pd.DataFrame(train_pca[tr_idx], columns=pca_names),
        ], axis=1)
        X_val = pd.concat([
            train_df[base_features].iloc[val_idx].reset_index(drop=True),
            pd.DataFrame(train_pca[val_idx], columns=pca_names),
        ], axis=1)
        model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
        model.fit(X_tr, y[tr_idx], eval_set=[(X_val, y[val_idx])], callbacks=[lgb.early_stopping(30, verbose=False)])
        scores.append(roc_auc_score(y[val_idx], model.predict_proba(X_val)[:, 1]))
    return float(np.mean(scores)), float(np.std(scores))

In [8]:
# 全組み合わせで実験（4 embedding × 2 pattern）。各 fold で「tr だけで PCA fit」してリーク防止。
results = []
base_features = list(BASELINE_FEATURES)

for config in EMBEDDING_CONFIGS:
    emb_df = load_embeddings(config)
    if emb_df is None:
        print(f"[SKIP] {config['name']}: ファイルなし", config["path"])
        for n_comp in PCA_PATTERNS:
            results.append({"embedding": config["name"], "pattern": f"PCA_{n_comp}",
                           "time_cv_auc_mean": np.nan, "time_cv_auc_std": np.nan,
                           "group_kfold_auc_mean": np.nan, "group_kfold_auc_std": np.nan})
        continue
    for n_comp in PCA_PATTERNS:
        prefix = f"emb_{config['name'][:12].replace(' ', '_')}_{n_comp}"
        print(f"[{config['name']}] PCA {n_comp} ...", end=" ")
        ts_m, ts_s = run_time_series_cv(train, test, base_features, y, time_splits, emb_df, n_comp, prefix)
        gk_m, gk_s = run_group_kfold_cv(train, test, base_features, y, group_splits, emb_df, n_comp, prefix)
        print(f"時系列CV={ts_m:.4f}±{ts_s:.4f}, GroupKFold={gk_m:.4f}±{gk_s:.4f}")
        results.append({
            "embedding": config["name"], "pattern": f"PCA_{n_comp}",
            "time_cv_auc_mean": ts_m, "time_cv_auc_std": ts_s,
            "group_kfold_auc_mean": gk_m, "group_kfold_auc_std": gk_s,
        })

results_df = pd.DataFrame(results)

[movie_info] PCA 8 ... 

KeyboardInterrupt: 

In [ ]:
# 結果の保存と表示
out_path = os.path.join(OUTPUT_DIR, "embedding_experiment_results.csv")
results_df.to_csv(out_path, index=False)
print(f"保存: {out_path}")
display(results_df)

## 提出ファイルのみ作成（学習・検証なし）

CV は行わず、各 embedding × PCA で **全 train で 1 本学習 → test 予測 → 提出 CSV 保存** だけ行う。  
パブリックスコアを見て方針を決める用。実行前に上からセル 1〜5（データ・設定）を実行しておく。

In [ ]:
# 提出用: 全 train で PCA fit → train/test を transform → 1 本学習 → 予測保存
SUBMISSIONS_DIR = os.path.join(OUTPUT_DIR, "submissions")
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

def save_submission(test_df, pred, path):
    pd.DataFrame({"ID": test_df["ID"], "target": pred}).to_csv(path, index=False)

base_features = list(BASELINE_FEATURES)
for config in EMBEDDING_CONFIGS:
    emb_df = load_embeddings(config)
    if emb_df is None:
        print(f"[SKIP] {config['name']}: ファイルなし → 提出スキップ")
        continue
    E_train = _merge_embeddings(train, emb_df)
    E_test = _merge_embeddings(test, emb_df)
    for n_comp in PCA_PATTERNS:
        n_comp = min(n_comp, E_train.shape[1])
        prefix = f"emb_{config['name'][:12].replace(' ', '_')}_{n_comp}"
        pca_names = [f"{prefix}_{i}" for i in range(n_comp)]
        pca = PCA(n_components=n_comp, random_state=42).fit(E_train)
        train_pca = pca.transform(E_train)
        test_pca = pca.transform(E_test)
        X_train = pd.concat([
            train[base_features].reset_index(drop=True),
            pd.DataFrame(train_pca, columns=pca_names),
        ], axis=1)
        X_test = pd.concat([
            test[base_features].reset_index(drop=True),
            pd.DataFrame(test_pca, columns=pca_names),
        ], axis=1)
        feats = base_features + pca_names
        model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
        model.fit(X_train[feats], y)
        pred = model.predict_proba(X_test[feats])[:, 1]
        assert len(pred) == len(test), "予測の行数が test と一致しません"
        safe_name = config["name"].replace(" ", "_")
        fname = f"submission_embedding_{safe_name}_pca{n_comp}.csv"
        out_path = os.path.join(SUBMISSIONS_DIR, fname)
        save_submission(test, pred, out_path)
        print(f"  [{config['name']}] PCA {n_comp} → {fname}")
print(f"\n保存先: {SUBMISSIONS_DIR}")

  [movie_info] PCA 8 → submission_embedding_movie_info_pca8.csv
  [movie_info] PCA 16 → submission_embedding_movie_info_pca16.csv
  [movie_info_large] PCA 8 → submission_embedding_movie_info_large_pca8.csv
  [movie_info_large] PCA 16 → submission_embedding_movie_info_large_pca16.csv
  [movie_title_info] PCA 8 → submission_embedding_movie_title_info_pca8.csv
  [movie_title_info] PCA 16 → submission_embedding_movie_title_info_pca16.csv
  [movie_title_info_large] PCA 8 → submission_embedding_movie_title_info_large_pca8.csv
  [movie_title_info_large] PCA 16 → submission_embedding_movie_title_info_large_pca16.csv

保存先: outputs/submissions


## バックグラウンドで実行する場合

ジム行きなどでノートブックを閉じる場合は、**ターミナル**で以下を実行するとバックグラウンドで同じ実験が回り、結果が `outputs/embedding_experiment_results.csv` に保存されます。

```bash
cd /Users/komatsuzakiharutoshi/Desktop/ds_dojo4
# 仮想環境を使う場合（.venv があるとき）
nohup .venv/bin/python run_embedding_experiments.py > outputs/embedding_experiments.log 2>&1 &
# または python3 が使える場合
# nohup python3 run_embedding_experiments.py > outputs/embedding_experiments.log 2>&1 &
```

ログ確認: `tail -f outputs/embedding_experiments.log`